In [1]:
## import 

import sys
# use line-buffering for both stdout and stderr
# sys.stdout = open(sys.stdout.fileno(), mode='w', buffering=1)
# sys.stderr = open(sys.stderr.fileno(), mode='w', buffering=1)

import hydra
from omegaconf import OmegaConf
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
import pathlib
import torch
import copy
import random
import wandb
import tqdm
import numpy as np
import shutil

from diffusion_policy.workspace.base_workspace import BaseWorkspace
from diffusion_policy.policy.robomimic_lowdim_policy import RobomimicLowdimPolicy
from diffusion_policy.dataset.base_dataset import BaseLowdimDataset
from diffusion_policy.env_runner.base_lowdim_runner import BaseLowdimRunner
from diffusion_policy.common.checkpoint_util import TopKCheckpointManager
from diffusion_policy.common.json_logger import JsonLogger
from diffusion_policy.common.pytorch_util import dict_apply, optimizer_to
from diffusion_policy.policy.robomimic_image_policy import RobomimicImagePolicy
from diffusion_policy.dataset.base_dataset import BaseImageDataset
from diffusion_policy.env_runner.base_image_runner import BaseImageRunner
from diffusion_policy.policy.diffusion_unet_hybrid_image_policy import DiffusionUnetHybridImagePolicy
from diffusion_policy.common.pytorch_util import dict_apply, optimizer_to
from diffusion_policy.model.diffusion.ema_model import EMAModel
from diffusion_policy.model.common.lr_scheduler import get_scheduler
from diffusion_policy.dataset.robomimic_replay_image_dataset import RobomimicReplayImageDataset

from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, Sampler 
import datetime
import h5py

/home/carl_lab/miniconda3/envs/robodiff/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [2]:
## Setup configuration

OmegaConf.register_new_resolver("eval", eval, replace=True)
config_path='diffusion_policy/config'
## should be in the diffusion_policy/config
config_name = "train_real_franka_pot"

with initialize(version_base=None, config_path=config_path):
    cfg_org = compose(
        config_name=config_name,
        overrides=[
            "hydra.run.dir=data/outputs/${now:%Y.%m.%d}/${now:%H.%M.%S}_${name}_${task_name}",
            "training.seed=42",
            "training.device=cuda:0"
        ],
    )
    print(cfg_org)
    
OmegaConf.resolve(cfg_org)

{'_target_': 'diffusion_policy.workspace.train_diffusion_unet_hybrid_workspace.TrainDiffusionUnetHybridWorkspace', 'shape_meta': {'action': {'shape': [10]}, 'obs': {'agentview_rgb': {'shape': [3, 240, 320], 'type': 'rgb'}, 'eye_in_hand_rgb': {'shape': [3, 240, 320], 'type': 'rgb'}, 'ee_states': {'shape': [16]}, 'joint_states': {'shape': [7]}, 'gripper_states': {'shape': [1]}}}, 'dataset_path': '/home/carl_lab/ola/stove_pot_bellpepper_lid_100/demo.hdf5', 'checkpoint': {'save_last_ckpt': True, 'save_last_snapshot': False, 'topk': {'format_str': 'epoch={epoch:04d}-test_mean_score={test_mean_score:.3f}.ckpt', 'k': 5, 'mode': 'max', 'monitor_key': 'test_mean_score'}}, 'dataloader': {'batch_size': 64, 'num_workers': 8, 'persistent_workers': False, 'pin_memory': True, 'shuffle': True}, 'val_dataloader': {'batch_size': 64, 'num_workers': 8, 'persistent_workers': False, 'pin_memory': True, 'shuffle': False}, 'dataset_obs_steps': 2, 'exp_name': 'default', 'horizon': 16, 'keypoint_visible_rate': 

In [ ]:
## Setup Workspace

class TrainDiffusionUnetHybridWorkspace(BaseWorkspace):
    include_keys = ['global_step', 'epoch']

    def __init__(self, cfg: OmegaConf, output_dir=None):
        super().__init__(cfg, output_dir=output_dir)

        # set seed
        seed = cfg.training.seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        # configure model
        self.model: DiffusionUnetHybridImagePolicy = hydra.utils.instantiate(cfg.policy)

        self.ema_model: DiffusionUnetHybridImagePolicy = None
        if cfg.training.use_ema:
            self.ema_model = copy.deepcopy(self.model)

        # configure training state
        self.optimizer = hydra.utils.instantiate(
            cfg.optimizer, params=self.model.parameters())

        # configure training state
        self.global_step = 0
        self.epoch = 0

timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
## location to save the checkpoints

# output_dir = f"/home/carl_lab/ola/diffusion_policy/data/outputs/custom{timestamp}"
output_dir = "/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55"


os.makedirs(output_dir, exist_ok=True)
print('output dir: ', output_dir)
workspace = TrainDiffusionUnetHybridWorkspace(cfg_org, output_dir=output_dir)

output dir:  /home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['gripper_states', 'ee_states', 'joint_states']
using obs modality: rgb with keys: ['agentview_rgb', 'eye_in_hand_rgb']
using obs modality: depth with keys: []
using obs modality: scan with keys: []


/home/carl_lab/miniconda3/envs/robodiff/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
/home/carl_lab/miniconda3/envs/robodiff/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Diffusion params: 2.564722e+08
Vision params: 2.239418e+07


In [ ]:
cfg = copy.deepcopy(workspace.cfg)

# resume training
if cfg.training.resume:
    lastest_ckpt_path = pathlib.Path("/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_1100_20260120_102355.ckpt")
    if lastest_ckpt_path.is_file():
        print(f"Resuming from checkpoint {lastest_ckpt_path}")
        workspace.load_checkpoint(path=lastest_ckpt_path)

new_config = OmegaConf.to_container(cfg.task.dataset, resolve=True )
del new_config['_target_'] ## remove from config



Resuming from checkpoint /home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_500_20260116_101410.ckpt


In [5]:
## Setup Dataset

dataset = RobomimicReplayImageDataset(**new_config)
print("dataset len", len(dataset))

cfg_dataloader = {key:value for key,value in cfg.dataloader.items()}

train_dataloader = DataLoader(dataset, **cfg_dataloader)
normalizer = dataset.get_normalizer()

# configure validation dataset
val_dataset = dataset.get_validation_dataset()
val_dataloader = DataLoader(val_dataset, **cfg.val_dataloader)

Acquiring lock on cache.
Loading cached ReplayBuffer from Disk.
Loaded!
dataset len 68320
Unsupported key: ee_states
Unsupported key: joint_states
Unsupported key: gripper_states


In [6]:
## Testing 
# batch = next(iter(train_dataloader))
# batch.keys()
# batch['action'].shape
# batch['obs']['agentview_rgb'].shape

In [6]:
batch = next(iter(train_dataloader))

In [7]:
batch.keys()

dict_keys(['obs', 'action'])

In [8]:
batch['action'].shape

torch.Size([64, 16, 10])

In [9]:
batch['obs']['agentview_rgb'].shape

torch.Size([64, 2, 3, 240, 320])

In [10]:
workspace.model.set_normalizer(normalizer)
if cfg.training.use_ema:
    print("Using EMA")
    workspace.ema_model.set_normalizer(normalizer)

# configure lr scheduler
lr_scheduler = get_scheduler(
    cfg.training.lr_scheduler,
    optimizer=workspace.optimizer,
    num_warmup_steps=cfg.training.lr_warmup_steps,
    num_training_steps=(
        len(train_dataloader) * cfg.training.num_epochs) \
            // cfg.training.gradient_accumulate_every,
    # pytorch assumes stepping LRScheduler every epoch
    # however huggingface diffusers steps it every batch
    last_epoch=workspace.global_step-1
)

# configure ema
ema: EMAModel = None
if cfg.training.use_ema:
    ema = hydra.utils.instantiate(
        cfg.ema,
        model=workspace.ema_model)


Using EMA


In [11]:
topk_manager = TopKCheckpointManager(
    save_dir=os.path.join(workspace.output_dir, 'checkpoints'),
    **cfg.checkpoint.topk
)

# device transfer
device = torch.device(cfg.training.device)
workspace.model.to(device)
if workspace.ema_model is not None:
    print("EMA is not none")
    workspace.ema_model.to(device)
optimizer_to(workspace.optimizer, device)

EMA is not none


AdamW (
Parameter Group 0
    amsgrad: False
    betas: [0.95, 0.999]
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 4.9879780855973903e-05
    maximize: False
    weight_decay: 1e-06
)

In [12]:
train_sampling_batch = None
log_path = os.path.join(workspace.output_dir, 'logs.json.txt')

with JsonLogger(log_path) as json_logger:
    for local_epoch_idx in range(cfg.training.num_epochs):
        step_log = dict()
        # ========= train for this epoch ==========
        train_losses = list()
        with tqdm.tqdm(train_dataloader, desc=f"Training epoch {workspace.epoch}", 
                leave=False, mininterval=cfg.training.tqdm_interval_sec) as tepoch:
            for batch_idx, batch in enumerate(tepoch):
                # device transfer
                batch = dict_apply(batch, lambda x: x.to(device, non_blocking=True))
                if train_sampling_batch is None:
                    train_sampling_batch = batch

                # compute loss
                raw_loss = workspace.model.compute_loss(batch)
                loss = raw_loss / cfg.training.gradient_accumulate_every
                loss.backward()

                # step optimizer
                if workspace.global_step % cfg.training.gradient_accumulate_every == 0:
                    workspace.optimizer.step()
                    workspace.optimizer.zero_grad()
                    lr_scheduler.step()
                
                # update ema
                if cfg.training.use_ema:
                    ema.step(workspace.model)

                # logging
                raw_loss_cpu = raw_loss.item()
                tepoch.set_postfix(loss=raw_loss_cpu, refresh=False)
                train_losses.append(raw_loss_cpu)
                step_log = {
                    'train_loss': raw_loss_cpu,
                    'global_step': workspace.global_step,
                    'epoch': workspace.epoch,
                    'lr': lr_scheduler.get_last_lr()[0]
                }

                is_last_batch = (batch_idx == (len(train_dataloader)-1))
                if not is_last_batch:
                    # log of last step is combined with validation and rollout
                     
                    json_logger.log(step_log)
                    workspace.global_step += 1

                if (cfg.training.max_train_steps is not None) \
                    and batch_idx >= (cfg.training.max_train_steps-1):
                    break

        # at the end of each epoch
        # replace train_loss with epoch average
        train_loss = np.mean(train_losses)
        step_log['train_loss'] = train_loss

        # ========= eval for this epoch ==========
        policy = workspace.model
        if cfg.training.use_ema:
            policy = workspace.ema_model
        policy.eval()

 
        # run validation
        if (workspace.epoch % cfg.training.val_every) == 0:
            with torch.no_grad():
                val_losses = list()
                with tqdm.tqdm(val_dataloader, desc=f"Validation epoch {workspace.epoch}", 
                        leave=False, mininterval=cfg.training.tqdm_interval_sec) as tepoch:
                    for batch_idx, batch in enumerate(tepoch):
                        batch = dict_apply(batch, lambda x: x.to(device, non_blocking=True))
                        loss = workspace.model.compute_loss(batch)
                        val_losses.append(loss)
                        if (cfg.training.max_val_steps is not None) \
                            and batch_idx >= (cfg.training.max_val_steps-1):
                            break
                if len(val_losses) > 0:
                    val_loss = torch.mean(torch.tensor(val_losses)).item()
                    # log epoch average validation loss
                    step_log['val_loss'] = val_loss

        # run diffusion sampling on a training batch
        if (workspace.epoch % cfg.training.sample_every) == 0:
            with torch.no_grad():
                # sample trajectory from training set, and evaluate difference
                batch = dict_apply(train_sampling_batch, lambda x: x.to(device, non_blocking=True))
                obs_dict = batch['obs']
                gt_action = batch['action']
                
                result = policy.predict_action(obs_dict)
                pred_action = result['action_pred']
                mse = torch.nn.functional.mse_loss(pred_action, gt_action)
                step_log['train_action_mse_error'] = mse.item()
                del batch
                del obs_dict
                del gt_action
                del result
                del pred_action
                del mse
        
        # checkpoint
        if (workspace.epoch % 100) == 0:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            # self.save_checkpoint(tag=f'epoch_{self.epoch}') 
            checkpoint_name = f'epoch_{workspace.epoch}_{timestamp}'
            path_ = workspace.save_checkpoint(tag=checkpoint_name)
            print(path_)

            
        # ========= eval end for this epoch ==========
        policy.train()

        # end of epoch
        # log of last step is combined with validation and rollout
         
        json_logger.log(step_log)
        workspace.global_step += 1
        workspace.epoch += 1



/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_500_20260118_152511.ckpt


/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_600_20260118_223621.ckpt


/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_700_20260119_054407.ckpt


/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_800_20260119_125304.ckpt


/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_900_20260119_200235.ckpt


/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_1000_20260120_031447.ckpt


/home/carl_lab/ola/diffusion_policy/data/outputs/custom2026_01_14_22_11_55/checkpoints/epoch_1100_20260120_102355.ckpt


KeyboardInterrupt: 

In [13]:
train_loss ## when stopped at 548: 0.002108539573588602

0.0006396378538966871

In [ ]:
workspace.save_checkpoint(tag=f"after_train_{workspace.epoch}_epochs")